<a href="https://colab.research.google.com/github/Vladislav-spect/DeepLearningSchool/blob/main/saving_loading_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

Сохранение и загрузка моделей
=========================

**Автор:** [Matthew Inkawhich](https://github.com/MatthewInkawhich)

Этот документ содержит решения для различных сценариев сохранения и загрузки моделей PyTorch. Можно прочитать документ целиком или сразу перейти к нужному разделу.

При сохранении и загрузке моделей необходимо знать три основные функции:

1)  [torch.save](https://pytorch.org/docs/stable/torch.html?highlight=save#torch.save):
    Сохраняет сериализованный объект на диск. Функция использует утилиту
    [pickle](https://docs.python.org/3/library/pickle.html) для сериализации.
    С её помощью можно сохранять модели, тензоры и словари любых объектов.
2)  [torch.load](https://pytorch.org/docs/stable/torch.html?highlight=torch%20load#torch.load):
    Использует возможности десериализации
    [pickle](https://docs.python.org/3/library/pickle.html) для загрузки объектов в память.
    Функция также позволяет указать устройство для загрузки данных (см.
    [Сохранение и загрузка модели на разных устройствах](#saving-loading-model-across-devices)).
3)  [torch.nn.Module.load\_state\_dict](https://pytorch.org/docs/stable/generated/torch.nn.Module.html?highlight=load_state_dict#torch.nn.Module.load_state_dict):
    Загружает словарь параметров модели из десериализованного *state\_dict*.
    Подробнее о *state\_dict* см. в разделе [Что такое state\_dict?](#what-is-a-state-dict).

**Содержание:**

-   [Что такое state\_dict?](#what-is-a-state-dict)
-   [Сохранение и загрузка модели для инференса](#saving-loading-model-for-inference)
-   [Сохранение и загрузка общего чекпоинта](#saving-loading-a-general-checkpoint-for-inference-and-or-resuming-training)
-   [Сохранение нескольких моделей в одном файле](#saving-multiple-models-in-one-file)
-   [Инициализация модели параметрами другой модели](#warmstarting-model-using-parameters-from-a-different-model)
-   [Сохранение и загрузка модели на разных устройствах](#saving-loading-model-across-devices)


Что такое `state_dict`?
=======================

В PyTorch обучаемые параметры (веса и смещения) модели `torch.nn.Module` хранятся в *параметрах* модели
(доступны через `model.parameters()`). *state\_dict* — это обычный словарь Python,
который сопоставляет каждый слой с его тензором параметров. Обратите внимание, что только слои
с обучаемыми параметрами (свёрточные, линейные и т.д.) и зарегистрированные буферы
(например, `running_mean` в BatchNorm) имеют записи в *state\_dict* модели.
Объекты оптимизатора (`torch.optim`) также имеют свой *state\_dict*, содержащий
информацию о состоянии оптимизатора и используемых гиперпараметрах.

Поскольку объекты *state\_dict* являются словарями Python, их можно легко
сохранять, обновлять, изменять и восстанавливать, что обеспечивает высокую
модульность моделей и оптимизаторов PyTorch.

Пример:
--------

Рассмотрим *state\_dict* простой модели из туториала
[Обучение классификатора](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#sphx-glr-beginner-blitz-cifar10-tutorial-py).

``` {.python}
# Define model
class TheModelClass(nn.Module):
    def __init__(self):
        super(TheModelClass, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize model
model = TheModelClass()

# Initialize optimizer
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Print model's state_dict
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

# Print optimizer's state_dict
print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])
```

**Вывод:**

``` {.sh}
Model's state_dict:
conv1.weight     torch.Size([6, 3, 5, 5])
conv1.bias   torch.Size([6])
conv2.weight     torch.Size([16, 6, 5, 5])
conv2.bias   torch.Size([16])
fc1.weight   torch.Size([120, 400])
fc1.bias     torch.Size([120])
fc2.weight   torch.Size([84, 120])
fc2.bias     torch.Size([84])
fc3.weight   torch.Size([10, 84])
fc3.bias     torch.Size([10])

Optimizer's state_dict:
state    {}
param_groups     [{'lr': 0.001, 'momentum': 0.9, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'params': [4675713712, 4675713784, 4675714000, 4675714072, 4675714216, 4675714288, 4675714432, 4675714504, 4675714648, 4675714720]}]
```


Те в model.state_dict() хранятся все обучаемые параметры (веса и смещения) каждого слоя модели, а также зарегистрированные буферы

Сохранение и загрузка модели для инференса
====================================

Сохранение/загрузка `state_dict` (рекомендуется)
------------------------------------

**Сохранение:**

``` {.python}
torch.save(model.state_dict(), PATH)
```

**Загрузка:**

``` {.python}
model = TheModelClass(*args, **kwargs)
model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()
```

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>ПРИМЕЧАНИЕ:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>В версии PyTorch 1.6 функция <code>torch.save</code> была переведена на новый формат на основе zip-файлов. <code>torch.load</code> по-прежнему поддерживает загрузку файлов старого формата. Если нужно использовать старый формат, передайте параметр <code>_use_new_zipfile_serialization=False</code>.</p>

</div>

При сохранении модели для инференса достаточно сохранить только обученные параметры.
Сохранение *state\_dict* с помощью `torch.save()` обеспечивает максимальную гибкость
при последующем восстановлении — именно поэтому этот метод рекомендуется.

Принятое соглашение в PyTorch — сохранять модели с расширением `.pt` или `.pth`.

Не забывайте вызывать `model.eval()` перед инференсом, чтобы перевести слои dropout
и batch normalization в режим оценки.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>ПРИМЕЧАНИЕ:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Функция <code>load_state_dict()</code> принимает объект-словарь, а НЕ путь к файлу. Это означает, что перед передачей необходимо десериализовать сохранённый *state_dict*. Нельзя писать <code>model.load_state_dict(PATH)</code>.</p>

</div>

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>ПРИМЕЧАНИЕ:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Если вы сохраняете только лучшую модель, помните: <code>best_model_state = model.state_dict()</code> возвращает ссылку, а не копию! Используйте <code>best_model_state = deepcopy(model.state_dict())</code>, иначе состояние будет обновляться с каждой итерацией обучения.</p>

</div>

Сохранение/загрузка всей модели
----------------------

**Сохранение:**

``` {.python}
torch.save(model, PATH)
```

**Загрузка:**

``` {.python}
# Класс модели должен быть определён в коде
model = torch.load(PATH, weights_only=False)
model.eval()
```

Этот способ использует наиболее интуитивный синтаксис и минимум кода. Модуль сериализуется
целиком с помощью [pickle](https://docs.python.org/3/library/pickle.html). Недостаток:
сохраняется не сам класс, а путь к файлу с ним — код может сломаться при переносе в другой
проект или после рефакторинга.

### Сохранение экспортированной программы

При использовании `torch.export` сохраняйте и загружайте `ExportedProgram`
через `torch.export.save()` и `torch.export.load()` с расширением `.pt2`:

``` {.python}
class SimpleModel(torch.nn.Module):
     def forward(self, x):
         return x + 10

sample_input = torch.randn(5)
exported_program = torch.export.export(SimpleModel(), sample_input)
torch.export.save(exported_program, 'exported_program.pt2')
saved_exported_program = torch.export.load('exported_program.pt2')
```


model = MyModel(input_size=784, hidden_size=128)

torch.save(model.state_dict(), 'model.pth')




А загрузка:

model = MyModel(input_size=784, hidden_size=128)   # ← вот здесь указываем архитектуру

model.load_state_dict(torch.load('model.pth', weights_only=True))

model.eval()



Если вы загрузили модель для инференса (то есть для получения предсказаний, а не для дальнейшего обучения), то обязательно нужно вызвать model.eval() после загрузки весов.

load_state_dict() — это метод, который ждёт на вход объект типа dict (словарь, где ключи — имена слоёв, значения — тензоры весов).

А torch.load('file.pth') читает файл и возвращает как раз такой словарь. Поэтому правильная цепочка:

python
model.load_state_dict(torch.load('model.pth', weights_only=True))


Нельзя писать:

python
model.load_state_dict('model.pth')   # ошибка: str не dict

В процессе дальнейшего обучения model меняет свои веса (например, делает градиентный шаг). Поскольку best_state просто ссылается на те же самые объекты, то best_state тоже изменится! Вы потеряете «лучшее» состояние.


Чтобы отделить сохранённое состояние от продолжающейся модели, нужно скопировать все тензоры. Это делает deepcopy:


from copy import deepcopy

best_state = deepcopy(model.state_dict())

Сохранение и загрузка общего чекпоинта для инференса и/или продолжения обучения
============================================================================

Сохранение:
-----

``` {.python}
torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
            ...
            }, PATH)
```

Загрузка:
-----

``` {.python}
model = TheModelClass(*args, **kwargs)
optimizer = TheOptimizerClass(*args, **kwargs)

checkpoint = torch.load(PATH, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']

model.eval()
# - или -
model.train()
```

При сохранении общего чекпоинта необходимо сохранять не только *state\_dict* модели,
но и *state\_dict* оптимизатора — он содержит буферы и параметры, обновляемые
в процессе обучения. Также полезно сохранить номер эпохи, последнее значение ошибки,
внешние слои `torch.nn.Embedding` и т.д. Такой чекпоинт обычно в 2–3 раза больше модели.

Организуйте данные в словарь и используйте `torch.save()` для сериализации.
Принятое соглашение — сохранять чекпоинты с расширением `.tar`.

Для загрузки сначала инициализируйте модель и оптимизатор, затем загрузите словарь
через `torch.load()`.

Вызывайте `model.eval()` перед инференсом или `model.train()` для продолжения обучения.


Сохранение нескольких моделей в одном файле
==================================

Сохранение:
-----

``` {.python}
torch.save({
            'modelA_state_dict': modelA.state_dict(),
            'modelB_state_dict': modelB.state_dict(),
            'optimizerA_state_dict': optimizerA.state_dict(),
            'optimizerB_state_dict': optimizerB.state_dict(),
            ...
            }, PATH)
```

Загрузка:
-----

``` {.python}
modelA = TheModelAClass(*args, **kwargs)
modelB = TheModelBClass(*args, **kwargs)
optimizerA = TheOptimizerAClass(*args, **kwargs)
optimizerB = TheOptimizerBClass(*args, **kwargs)

checkpoint = torch.load(PATH, weights_only=True)
modelA.load_state_dict(checkpoint['modelA_state_dict'])
modelB.load_state_dict(checkpoint['modelB_state_dict'])
optimizerA.load_state_dict(checkpoint['optimizerA_state_dict'])
optimizerB.load_state_dict(checkpoint['optimizerB_state_dict'])

modelA.eval()
modelB.eval()
# - или -
modelA.train()
modelB.train()
```

При сохранении модели из нескольких `torch.nn.Module` (GAN, sequence-to-sequence и т.д.)
применяется тот же подход, что и для общего чекпоинта: сохраните словарь с *state\_dict*
каждой модели и оптимизатора. Принятое соглашение — расширение `.tar`.

Для загрузки инициализируйте модели и оптимизаторы, затем загрузите словарь через `torch.load()`.

Вызывайте `model.eval()` перед инференсом или `model.train()` для продолжения обучения.


*args и **kwargs — это специальный синтаксис Python, который означает «все остальные позиционные аргументы» (*args) и «все именованные аргументы» (**kwargs).

Инициализация модели параметрами другой модели
==========================================================

Сохранение:
-----

``` {.python}
torch.save(modelA.state_dict(), PATH)
```

Загрузка:
-----

``` {.python}
modelB = TheModelBClass(*args, **kwargs)
modelB.load_state_dict(torch.load(PATH, weights_only=True), strict=False)
```

Частичная загрузка модели — распространённый сценарий при трансферном обучении.
Использование уже обученных параметров ускорит сходимость по сравнению с обучением с нуля.

Если *state\_dict* содержит лишние или недостающие ключи, установите `strict=False`
в `load_state_dict()`, чтобы игнорировать несовпадающие ключи.

Если ключи не совпадают — переименуйте нужные ключи в загружаемом *state\_dict*.


Сохранение и загрузка модели на разных устройствах
=====================================

Сохранение на GPU, загрузка на CPU
------------------------

**Сохранение:**

``` {.python}
torch.save(model.state_dict(), PATH)
```

**Загрузка:**

``` {.python}
device = torch.device('cpu')
model = TheModelClass(*args, **kwargs)
model.load_state_dict(torch.load(PATH, map_location=device, weights_only=True))
```

При загрузке модели на CPU передайте `torch.device('cpu')` в аргумент `map_location`.

Сохранение на GPU, загрузка на GPU
------------------------

**Сохранение:**

``` {.python}
torch.save(model.state_dict(), PATH)
```

**Загрузка:**

``` {.python}
device = torch.device("cuda")
model = TheModelClass(*args, **kwargs)
model.load_state_dict(torch.load(PATH, weights_only=True))
model.to(device)
# Не забудьте: input = input.to(device) для всех входных тензоров
```

Преобразуйте модель с помощью `model.to(torch.device('cuda'))`. Помните:
`my_tensor.to(device)` создаёт новую копию — перезаписывайте переменную:
`my_tensor = my_tensor.to(torch.device('cuda'))`.

Сохранение на CPU, загрузка на GPU
------------------------

**Сохранение:**

``` {.python}
torch.save(model.state_dict(), PATH)
```

**Загрузка:**

``` {.python}
device = torch.device("cuda")
model = TheModelClass(*args, **kwargs)
model.load_state_dict(torch.load(PATH, weights_only=True, map_location="cuda:0"))
model.to(device)
# Не забудьте: input = input.to(device) для всех входных тензоров
```

Укажите `map_location="cuda:device_id"`, затем вызовите `model.to(torch.device('cuda'))`.

Сохранение моделей `torch.nn.DataParallel`
-------------------------------------

**Сохранение:**

``` {.python}
torch.save(model.module.state_dict(), PATH)
```

**Загрузка:**

``` {.python}
# Загрузите на любое устройство по вашему выбору
```

`torch.nn.DataParallel` — обёртка для параллельного использования GPU.
Используйте `model.module.state_dict()` для универсального сохранения.
